# M6 · Combined optimizations and the independent quality gate

**Question:** Is the fastest configuration automatically the best deployment? Compare B0 to BEST, distinguish LPIPS perceptual distance from fidelity, and preserve BEST's independent PENDING_HUMAN_REVIEW verdict.

**Learning loop:** predict → inspect frozen evidence → compute → critique → propose a controlled experiment.

Reference lesson: [open course/M06_combined_quality/LESSON.md](../course/M06_combined_quality/LESSON.md). Switch language: [中文](./07_combined_and_quality.zh-CN.ipynb).

**Default mode:** CPU-only, internet-free, read-only. GPU work is never run implicitly.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## Inspect and recompute real evidence

Run cells in order. Explain every denominator and scope.

In [ ]:
rows = ns.summary_rows()
b, best = rows["B0"], rows["BEST"]
ratio = b["warm_total_median_s"] / best["warm_total_median_s"]
print(f"Measured warm B0={b['warm_total_median_s']:.3f}s; BEST={best['warm_total_median_s']:.3f}s; gain={ratio:.3f}x")
assert 6.0 < ratio < 6.3
print("BEST changes model variant, schedule AND precision. This is not a lossless kernel swap.")
plot_values = {"B0": b["warm_total_median_s"], "BEST": best["warm_total_median_s"]}
plot_ylabel, plot_title = "warm seconds", "Combined result requires separate image review"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


In [ ]:
quality = ns.read_json("evidence/quality/BEST/quality.json")
assert quality["semantics"] == "distance"
print("LPIPS is a DISTANCE here, not an objective quality grade.")
for prompt_id, value in quality["median"].items():
    print(f"{prompt_id}: {value:.3f} (distance)")
claims = ns.read_json("course/data/verified_claims.json")["claims"]
pending = [x for x in claims if "BEST" in x["id"] and "PENDING" in str(x.get("human_quality_verdict", ""))]
assert pending, "An independent BEST human-quality review is required"
print("BEST quality gate is PENDING; do not mark PASS automatically.")


In [ ]:
# Human-readable side-by-side visual review is optional in a real Jupyter kernel.
SHOW_QUALITY_IMAGES = False
if SHOW_QUALITY_IMAGES:
    from IPython.display import display, Image
    display(Image(filename=str(ROOT / "evidence/quality/BEST/quality_grid.png")))
    display(Image(filename=str(ROOT / "evidence/quality/BEST/quality_crops_poster.png")))
else:
    print("Image display is OFF. Set SHOW_QUALITY_IMAGES=True to inspect grids/native poster crops.")
print("Looking at images does NOT automatically create an owner verdict.")


## Optional physical GPU smoke — explicit opt-in

Requires the staged ROCm environment and sufficient model/GPU resources. Never used in CPU CI. The run goes to `runs/`, not official evidence.

In [ ]:
RUN_GPU = False  # Only switch on after checking platform quota + available models
if RUN_GPU:
    # Also set QIR_NOTEBOOK_GPU_ACK=YES in your own kernel environment.
    # The helper enforces --smoke and writes under runs/, never evidence/.
    ns.gpu_smoke("BEST", "notebook-best", confirmed=True)
else:
    print("GPU NOT_RUN; evidence analysis above is valid without GPU access.")


## Your turn / Self-check

Inspect the poster text at native resolution in `evidence/quality/BEST/quality_crops_poster.png`. Write a proposed three-level manual rubric; this notebook NEVER issues an owner verdict.

**Evidence discipline:** Answer in your own words. This notebook's numbers come from existing committed records, not your GPU. If your optional smoke runs, its data must remain under `runs/` and may not be presented as a full-protocol median.
